# Revisions R2.1 and R2.4: annotation reliability and annotation scope (no GPU)

**Reviewer 2, comment 1:** *"Agreement was measured on only 100 queries … using token-level kappa, which is inflated by O tokens."*
→ Entity-level agreement on the doubly annotated tasks: exact span F1, overlapping-span F1, κ on all tokens and on entity tokens only.

**Reviewer 2, comment 4:** *"Explain whether [medication] entities occur in the source queries … and which clinically relevant information (including negated entities, currently labeled O) is outside the annotation task."*
→ Counts of medication, procedure and negated mentions in all 1,742 queries.

These were already computed once (results are in the response letter). This notebook makes them reproducible for the repository.

**Extra file needed:** the Label Studio export, saved as `MyDrive/strokeid_revision/data/export.json`.

**Runtime:** no GPU needed. Choose **CPU** (*Runtime → Change runtime type → CPU*) to save compute units.

In [ ]:
!pip install -q pytorch-crf seqeval openai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
PROJECT = '/content/drive/MyDrive/strokeid_revision'   # <- same folder in every notebook
DATA, RUNS, RESULTS, CODE = f'{PROJECT}/data', f'{PROJECT}/runs', f'{PROJECT}/results', '/content/code'
for d in (RUNS, RESULTS, f'{CODE}/prompts'): os.makedirs(d, exist_ok=True)
sys.path.insert(0, CODE)
for f in ('train.tsv', 'val.tsv', 'test.tsv'):
    assert os.path.exists(f'{DATA}/{f}'), f'missing {DATA}/{f} (copies of the *_fixed.tsv files)'
print('setup OK')

In [ ]:
%%writefile /content/code/common.py
"""Shared helpers for the StrokeID-NER revision experiments.

Data format (one of):
  * CoNLL/TSV: one "token<TAB>label" per line, blank line between queries.
  * JSONL: one {"id": ..., "tokens": [...], "labels": [...]} object per line.
"""
import json
import os
from collections import Counter

ENTITY_TYPES = ["DGN", "RF", "SYM", "TMP"]
LABELS = ["O", "B-SYM", "I-SYM", "B-DGN", "I-DGN", "B-TMP", "I-TMP", "B-RF", "I-RF"]


def load_split(path):
    """Return a list of dicts: {id, tokens, labels}."""
    if path.endswith(".jsonl"):
        out = []
        with open(path, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if line.strip():
                    d = json.loads(line)
                    d.setdefault("id", i)
                    out.append(d)
        return out
    out, toks, labs = [], [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                if toks:
                    out.append({"id": len(out), "tokens": toks, "labels": labs})
                    toks, labs = [], []
                continue
            parts = line.split("\t") if "\t" in line else line.split()
            toks.append(parts[0])
            labs.append(parts[-1])
    if toks:
        out.append({"id": len(out), "tokens": toks, "labels": labs})
    return out


def find_split(data_dir, name):
    for ext in (".jsonl", ".tsv", ".conll", ".txt"):
        p = os.path.join(data_dir, name + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"No {name}.(jsonl|tsv|conll|txt) in {data_dir}")


def get_spans(labels):
    """Extract (type, start, end_inclusive) spans from a BIO sequence.

    Mirrors seqeval's default (non-strict) behaviour: an I- tag that does not
    continue a span of the same type starts a new span.
    """
    spans, cur = [], None
    for i, lab in enumerate(list(labels) + ["O"]):
        tag, typ = (lab.split("-", 1) + [None])[:2] if lab != "O" else ("O", None)
        if cur is not None and (tag in ("O", "B") or typ != cur[0]):
            spans.append((cur[0], cur[1], i - 1))
            cur = None
        if tag == "B" or (tag == "I" and cur is None):
            cur = (typ, i)
    return spans


def per_query_counts(gold_seqs, pred_seqs, types=ENTITY_TYPES):
    """Array-like counts [n_queries][type] -> (tp, n_pred, n_gold), exact match."""
    rows = []
    for g, p in zip(gold_seqs, pred_seqs):
        gs, ps = set(get_spans(g)), set(get_spans(p))
        row = []
        for t in types:
            gt = {s for s in gs if s[0] == t}
            pt = {s for s in ps if s[0] == t}
            row.append((len(gt & pt), len(pt), len(gt)))
        rows.append(row)
    return rows


def macro_from_totals(tp, npred, ngold):
    """tp/npred/ngold are per-type sequences. Returns macro P, R, F1 and per-type F1
    (seqeval 'macro avg' definition: unweighted mean of per-type scores)."""
    ps, rs, fs = [], [], []
    for a, b, c in zip(tp, npred, ngold):
        p = a / b if b else 0.0
        r = a / c if c else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        ps.append(p); rs.append(r); fs.append(f)
    n = len(fs)
    return sum(ps) / n, sum(rs) / n, sum(fs) / n, fs


def write_predictions(path, examples, preds):
    with open(path, "w", encoding="utf-8") as f:
        for ex, p in zip(examples, preds):
            assert len(p) == len(ex["tokens"]), (ex["id"], len(p), len(ex["tokens"]))
            f.write(json.dumps({"id": ex["id"], "tokens": ex["tokens"],
                                "gold": ex["labels"], "pred": p}, ensure_ascii=False) + "\n")


def read_predictions(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def label_stats(examples):
    c = Counter()
    for ex in examples:
        for t, *_ in get_spans(ex["labels"]):
            c[t] += 1
    return dict(c)


In [ ]:
%%writefile /content/code/iaa_from_labelstudio.py
"""Extract the doubly annotated tasks from the Label Studio export (R2.1).

Tasks with two annotations are the inter-annotator sample. The older annotation
is the primary annotator, the newer one the second annotator (both stored under
the same Label Studio account; see Methods). Text is tokenised with the regex
\\w+|[^\\w\\s], which reproduces the submitted token-level kappa (0.857).

Usage:
  python iaa_from_labelstudio.py --export export.json --out_dir results/iaa
  python iaa.py --a results/iaa/primary.jsonl --b results/iaa/second.jsonl --out results/iaa/iaa.json
"""
import argparse
import json
import os
import re

TOKEN = re.compile(r"\w+|[^\w\s]")


def to_bio(text, spans):
    toks = list(TOKEN.finditer(text))
    labels = []
    for m in toks:
        lab = "O"
        for typ, s, e in spans:
            if s <= m.start() < e:
                inside_before = any(s <= mm.start() < m.start() for mm in toks)
                lab = ("I-" if inside_before else "B-") + typ
        labels.append(lab)
    return [m.group() for m in toks], labels


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--export", required=True)
    ap.add_argument("--out_dir", required=True)
    a = ap.parse_args()
    os.makedirs(a.out_dir, exist_ok=True)
    tasks = [t for t in json.load(open(a.export, encoding="utf-8")) if len(t["annotations"]) == 2]
    with open(os.path.join(a.out_dir, "primary.jsonl"), "w", encoding="utf-8") as fa, \
         open(os.path.join(a.out_dir, "second.jsonl"), "w", encoding="utf-8") as fb:
        for t in tasks:
            first, second = sorted(t["annotations"], key=lambda x: x["created_at"])
            for ann, f in ((first, fa), (second, fb)):
                spans = [(r["value"]["labels"][0], r["value"]["start"], r["value"]["end"])
                         for r in ann["result"] if "labels" in r.get("value", {})]
                toks, labs = to_bio(t["data"]["text"], spans)
                f.write(json.dumps({"id": t["id"], "tokens": toks, "labels": labs}, ensure_ascii=False) + "\n")
    print(f"{len(tasks)} doubly annotated tasks written to {a.out_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/code/iaa.py
"""Inter-annotator agreement beyond token-level kappa (R2.1).

Reports, on the doubly annotated sample:
  * entity-level (span) agreement F1, exact match of boundaries + type
    (Hripcsak & Rothschild 2005: pairwise F1 approximates kappa for NER when
    the number of negative cases is undefined) - overall micro, macro, per type;
  * relaxed agreement F1 (same type, overlapping spans);
  * token-level Cohen's kappa on all tokens (the number currently reported) and
    on tokens where at least one annotator assigned an entity label (removes the
    inflation from O-O agreement the reviewer points out).

Usage: python iaa.py --a primary_100.jsonl --b second_100.jsonl [--out results/iaa.json]
Both files: same queries, same tokenisation (JSONL with tokens/labels or CoNLL).
"""
import argparse
import json
from collections import Counter

from common import ENTITY_TYPES, get_spans, load_split


def kappa(x, y):
    n = len(x)
    po = sum(a == b for a, b in zip(x, y)) / n
    cx, cy = Counter(x), Counter(y)
    pe = sum(cx[k] * cy[k] for k in set(cx) | set(cy)) / (n * n)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0


def f1(tp, na, nb):
    return 2 * tp / (na + nb) if na + nb else 0.0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--a", required=True); ap.add_argument("--b", required=True)
    ap.add_argument("--out", default=None)
    o = ap.parse_args()
    A, B = load_split(o.a), load_split(o.b)
    assert len(A) == len(B), "different number of queries"
    exact, relaxed, na, nb = Counter(), Counter(), Counter(), Counter()
    tok_a, tok_b = [], []
    disagreements = []
    for qa, qb in zip(A, B):
        assert qa["tokens"] == qb["tokens"], f"tokenisation differs in query {qa['id']}"
        tok_a += qa["labels"]; tok_b += qb["labels"]
        sa, sb = set(get_spans(qa["labels"])), set(get_spans(qb["labels"]))
        for t in ENTITY_TYPES:
            ta = [s for s in sa if s[0] == t]; tb = [s for s in sb if s[0] == t]
            na[t] += len(ta); nb[t] += len(tb)
            exact[t] += len(set(ta) & set(tb))
            # relaxed: greedy one-to-one matching of overlapping same-type spans
            used = set()
            for s in ta:
                for j, u in enumerate(tb):
                    if j not in used and not (u[2] < s[1] or u[1] > s[2]):
                        used.add(j); relaxed[t] += 1; break
        if sa != sb:
            disagreements.append({"id": qa["id"], "tokens": qa["tokens"],
                                  "only_a": sorted(sa - sb), "only_b": sorted(sb - sa)})
    per_exact = {t: f1(exact[t], na[t], nb[t]) for t in ENTITY_TYPES}
    per_relax = {t: f1(relaxed[t], na[t], nb[t]) for t in ENTITY_TYPES}
    ent_idx = [i for i, (x, y) in enumerate(zip(tok_a, tok_b)) if x != "O" or y != "O"]
    res = dict(
        n_queries=len(A), n_tokens=len(tok_a), spans_a=dict(na), spans_b=dict(nb),
        exact_f1_micro=f1(sum(exact.values()), sum(na.values()), sum(nb.values())),
        exact_f1_macro=sum(per_exact.values()) / len(ENTITY_TYPES), exact_f1_per_type=per_exact,
        relaxed_f1_micro=f1(sum(relaxed.values()), sum(na.values()), sum(nb.values())),
        relaxed_f1_per_type=per_relax,
        token_kappa_all=kappa(tok_a, tok_b),
        token_kappa_entity_tokens_only=kappa([tok_a[i] for i in ent_idx], [tok_b[i] for i in ent_idx]),
        n_entity_tokens=len(ent_idx),
        share_tokens_both_O=1 - len(ent_idx) / len(tok_a),
        n_queries_with_disagreement=len(disagreements))
    print(json.dumps(res, indent=2))
    if o.out:
        json.dump(dict(res, disagreements=disagreements), open(o.out, "w"), indent=2, ensure_ascii=False)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/code/scope_counts.py
"""Corpus counts for the annotation-scope rationale (R2.4).

Finds, across all splits:
  * medication / prescription mentions (lexicon + dosage patterns),
  * procedure / examination mentions (out of scope per guideline),
  * negated mentions: a negation cue followed (within 3 tokens) by a word that is
    annotated as an entity elsewhere in the corpus but is labelled O here.
Every hit is written out with its query so you can VERIFY BY HAND before quoting
a number in the paper - lexicon matching over- and under-counts.

Usage: python scope_counts.py --data_dir data --out results/scope
"""
import argparse
import json
import os
import re
from collections import Counter, defaultdict

from common import find_split, get_spans, load_split

MEDS = {
    # antiplatelet / anticoagulant / thrombolytic
    "aspirin", "asetosal", "aspilet", "miniaspi", "clopidogrel", "cpg", "plavix", "cilostazol",
    "warfarin", "heparin", "rivaroxaban", "dabigatran", "alteplase", "rtpa", "rt-pa",
    # neuro "supplements" common in Indonesia
    "citicoline", "citicolin", "sitikolin", "piracetam", "neurobion", "mecobalamin", "mecobalamine",
    "betahistine", "flunarizine", "gabapentin", "pregabalin",
    # antihypertensives
    "amlodipin", "amlodipine", "captopril", "lisinopril", "ramipril", "candesartan", "valsartan",
    "losartan", "irbesartan", "telmisartan", "bisoprolol", "propranolol", "furosemide", "hct", "nifedipin",
    "nifedipine", "nicardipine", "clonidine", "spironolactone",
    # lipid / diabetes
    "simvastatin", "atorvastatin", "rosuvastatin", "metformin", "glimepiride", "glibenclamide", "insulin",
    # analgesic / other
    # "asam" deliberately excluded: "asam urat" (uric acid) is an RF
    "paracetamol", "ibuprofen", "mefenamat", "diazepam", "phenytoin", "fenitoin",
    "omeprazole", "lansoprazole", "ranitidin", "dexamethasone", "prednison",
}
GENERIC_MED = {"obat", "obatnya", "obat-obatan", "tablet", "kapsul", "resep", "suntik", "suntikan",
               "infus", "jamu", "herbal", "suplemen", "vitamin"}
DOSE = re.compile(r"^\d+(mg|ml|mcg|gr|g|iu)$", re.I)
PROC = {"ct", "ct-scan", "ctscan", "mri", "eeg", "ekg", "ecg", "rontgen", "operasi", "fisioterapi",
        "terapi", "cek", "lab", "laboratorium", "rawat", "opname", "kraniotomi", "angiografi", "dsa"}
NEG = {"tidak", "tdk", "tak", "bukan", "belum", "blm", "tanpa", "gak", "ga", "nggak", "ngga", "enggak",
       "engga", "gk", "g", "ndak", "nda"}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True); ap.add_argument("--out", default="results/scope")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    data = []
    for s in ("train", "val", "test"):
        data += [dict(q, split=s) for q in load_split(find_split(a.data_dir, s))]
    N = len(data)

    # "entity words": words that START an entity span at least twice and are inside
    # an entity in at least half of all their occurrences (filters out function
    # words such as "saya", "pada" that happen to sit inside some spans).
    ent_words = defaultdict(Counter)
    starts, total = Counter(), Counter()
    for q in data:
        for w in q["tokens"]:
            total[w.lower()] += 1
        for t, i, j in get_spans(q["labels"]):
            starts[q["tokens"][i].lower()] += 1
            for w in q["tokens"][i:j + 1]:
                ent_words[w.lower()][t] += 1
    ent_words = {w: c for w, c in ent_words.items()
                 if starts[w] >= 2 and sum(c.values()) / total[w] >= 0.5 and len(w) > 2 and w not in NEG}

    hits = {"medication_specific": [], "medication_generic": [], "dosage": [], "procedure": [], "negated": []}
    q_with = {k: set() for k in hits}
    term_counts = {k: Counter() for k in hits}
    med_label = Counter()
    for q in data:
        toks = [t.lower() for t in q["tokens"]]
        for i, (w, lab) in enumerate(zip(toks, q["labels"])):
            for key, cond in (("medication_specific", w in MEDS), ("medication_generic", w in GENERIC_MED),
                              ("dosage", bool(DOSE.match(w))), ("procedure", w in PROC)):
                if cond:
                    hits[key].append(dict(id=q["id"], split=q["split"], i=i, token=q["tokens"][i], label=lab,
                                          context=" ".join(q["tokens"][max(0, i - 6):i + 7])))
                    q_with[key].add((q["split"], q["id"])); term_counts[key][w] += 1
                    if key == "medication_specific":
                        med_label[lab.split("-")[-1] if lab != "O" else "O"] += 1
            if w in NEG:
                for k in range(i + 1, min(i + 4, len(toks))):
                    if q["labels"][k] == "O" and toks[k] in ent_words and sum(ent_words[toks[k]].values()) >= 2:
                        typ = ent_words[toks[k]].most_common(1)[0][0]
                        hits["negated"].append(dict(id=q["id"], split=q["split"], cue=q["tokens"][i],
                                                    word=q["tokens"][k], usual_type=typ,
                                                    context=" ".join(q["tokens"][max(0, i - 4):k + 5])))
                        q_with["negated"].add((q["split"], q["id"])); term_counts["negated"][typ] += 1
                        break
    summary = {k: dict(mentions=len(v), queries=len(q_with[k]), pct_queries=round(100 * len(q_with[k]) / N, 1),
                       top_terms=term_counts[k].most_common(15)) for k, v in hits.items()}
    summary["n_queries"] = N
    summary["medication_specific_label_distribution"] = dict(med_label)
    json.dump(summary, open(os.path.join(a.out, "scope_summary.json"), "w"), indent=2, ensure_ascii=False)
    for k, v in hits.items():
        with open(os.path.join(a.out, f"hits_{k}.jsonl"), "w") as f:
            for h in v:
                f.write(json.dumps(h, ensure_ascii=False) + "\n")
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print(f"\nHit lists for manual verification written to {a.out}/hits_*.jsonl")


if __name__ == "__main__":
    main()


## R2.1: inter-annotator agreement
Expected: 103 tasks, token κ 0.857 (reproduces the paper), exact entity F1 0.781, overlapping F1 0.960.

In [ ]:
!cd /content/code && python iaa_from_labelstudio.py --export "$DATA/export.json" --out_dir "$RESULTS/iaa"
!cd /content/code && python iaa.py --a "$RESULTS/iaa/primary.jsonl" --b "$RESULTS/iaa/second.jsonl" --out "$RESULTS/iaa/iaa.json"

## R2.4: annotation scope
Word-list matching, so check the hit files by hand before quoting a number (`results/scope/hits_*.jsonl`). The letter uses the manually verified negation count (about 21).

In [ ]:
!cd /content/code && python scope_counts.py --data_dir "$DATA" --out "$RESULTS/scope"